## Single Layer DNN Models

In [2]:
import os
import time
import warnings
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn as sk
import tensorflow as tf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# set current data path
data_path = os.getcwd()
output_dir = Path(data_path) / "single_layer_results"
output_dir.mkdir(exist_ok=True)

# import kaggle token
token = Path("kaggletoken.txt").read_text().strip()
os.environ["KAGGLE_API_TOKEN"] = token

# define kaggle download
download_path = kagglehub.dataset_download(
    "vincentvaseghi/us-cities-housing-market-data",
    path="city_market_tracker.tsv000",
    output_dir=data_path + "/data",
)

# check rocm devices
devices = tf.config.list_physical_devices("GPU")
print("GPUs found:", devices)
print("Saving outputs to:", output_dir)


/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0000 00:00:1773936746.085909      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085976      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085979      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773936746.085981      27 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-03-19 12:12:26.095628: I tensorflow/core/platform/cpu_f

GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Import into pandas

In [3]:
import pandas as pd

# load the tsv as a csv but with tab seperators
df = pd.read_csv(data_path + "/data/city_market_tracker.tsv000", sep="\t")

# print head of dataset
print(df.head())
df.columns.tolist()

  PERIOD_BEGIN  PERIOD_END  PERIOD_DURATION REGION_TYPE  REGION_TYPE_ID  \
0   2021-08-01  2021-08-31               30       place               6   
1   2022-03-01  2022-03-31               30       place               6   
2   2016-09-01  2016-09-30               30       place               6   
3   2025-12-01  2025-12-31               30       place               6   
4   2018-03-01  2018-03-31               30       place               6   

   TABLE_ID  IS_SEASONALLY_ADJUSTED            REGION          CITY  \
0     14689                   False   St. Augusta, MN   St. Augusta   
1     30390                   False       Newfane, NY       Newfane   
2     18083                   False  Simpsonville, KY  Simpsonville   
3     11261                   False         Maize, KS         Maize   
4     30755                   False    Loganville, GA    Loganville   

       STATE  ... SOLD_ABOVE_LIST_YOY PRICE_DROPS  PRICE_DROPS_MOM  \
0  Minnesota  ...            0.354167    0.272727   

['PERIOD_BEGIN',
 'PERIOD_END',
 'PERIOD_DURATION',
 'REGION_TYPE',
 'REGION_TYPE_ID',
 'TABLE_ID',
 'IS_SEASONALLY_ADJUSTED',
 'REGION',
 'CITY',
 'STATE',
 'STATE_CODE',
 'PROPERTY_TYPE',
 'PROPERTY_TYPE_ID',
 'MEDIAN_SALE_PRICE',
 'MEDIAN_SALE_PRICE_MOM',
 'MEDIAN_SALE_PRICE_YOY',
 'MEDIAN_LIST_PRICE',
 'MEDIAN_LIST_PRICE_MOM',
 'MEDIAN_LIST_PRICE_YOY',
 'MEDIAN_PPSF',
 'MEDIAN_PPSF_MOM',
 'MEDIAN_PPSF_YOY',
 'MEDIAN_LIST_PPSF',
 'MEDIAN_LIST_PPSF_MOM',
 'MEDIAN_LIST_PPSF_YOY',
 'HOMES_SOLD',
 'HOMES_SOLD_MOM',
 'HOMES_SOLD_YOY',
 'PENDING_SALES',
 'PENDING_SALES_MOM',
 'PENDING_SALES_YOY',
 'NEW_LISTINGS',
 'NEW_LISTINGS_MOM',
 'NEW_LISTINGS_YOY',
 'INVENTORY',
 'INVENTORY_MOM',
 'INVENTORY_YOY',
 'MONTHS_OF_SUPPLY',
 'MONTHS_OF_SUPPLY_MOM',
 'MONTHS_OF_SUPPLY_YOY',
 'MEDIAN_DOM',
 'MEDIAN_DOM_MOM',
 'MEDIAN_DOM_YOY',
 'AVG_SALE_TO_LIST',
 'AVG_SALE_TO_LIST_MOM',
 'AVG_SALE_TO_LIST_YOY',
 'SOLD_ABOVE_LIST',
 'SOLD_ABOVE_LIST_MOM',
 'SOLD_ABOVE_LIST_YOY',
 'PRICE_DROPS',
 'PRICE_DRO

## Clean the data

In [19]:
# drop na data
df = df.dropna().copy()

# convert period start to real date time
df["PERIOD_BEGIN"] = pd.to_datetime(df["PERIOD_BEGIN"])
df = df.sort_values("PERIOD_BEGIN").reset_index(drop=True)

# add new columns for dates
df["MONTH"] = df["PERIOD_BEGIN"].dt.month
df["YEAR"] = df["PERIOD_BEGIN"].dt.year

# seasonal features
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)

# lag features by region
df["PRICE_LAG_1"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(1)
df["PRICE_LAG_3"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(3)

df = df.drop(columns=["PERIOD_END", "PERIOD_DURATION", "TABLE_ID"])
df = df.dropna().copy()

# encode string columns
categorical_cols = df.select_dtypes(include=["object"]).columns
oe = sk.preprocessing.OrdinalEncoder()
df[categorical_cols] = oe.fit_transform(df[categorical_cols])


KeyError: "['PERIOD_END', 'PERIOD_DURATION', 'TABLE_ID'] not found in axis"

## Pre Process before Training

In [20]:
feature_cols = [
    "STATE",
    "CITY",
    "PROPERTY_TYPE",
    "INVENTORY",
    "REGION",
    "HOMES_SOLD",
    "MEDIAN_DOM",
    "AVG_SALE_TO_LIST",
    "PRICE_LAG_1",
    "PRICE_LAG_3",
    "MONTH_SIN",
    "MONTH_COS",
    "YEAR",
]
target_col = "MEDIAN_SALE_PRICE"

# keep the last 3 years for test and the prior 2 years for validation
cutoff_test = df["PERIOD_BEGIN"].max() - pd.DateOffset(years=3)
cutoff_val = cutoff_test - pd.DateOffset(years=2)

train_df = df[df["PERIOD_BEGIN"] < cutoff_val].copy()
val_df = df[(df["PERIOD_BEGIN"] >= cutoff_val) & (df["PERIOD_BEGIN"] < cutoff_test)].copy()
test_df = df[df["PERIOD_BEGIN"] >= cutoff_test].copy()

scaler = sk.preprocessing.MinMaxScaler()
x_train = pd.DataFrame(
    scaler.fit_transform(train_df[feature_cols]),
    columns=feature_cols,
    index=train_df.index,
)
x_val = pd.DataFrame(
    scaler.transform(val_df[feature_cols]),
    columns=feature_cols,
    index=val_df.index,
)
x_test = pd.DataFrame(
    scaler.transform(test_df[feature_cols]),
    columns=feature_cols,
    index=test_df.index,
)

y_train = train_df[target_col].astype("float32")
y_val = val_df[target_col].astype("float32")
y_test = test_df[target_col].astype("float32")

print("Train shape:", x_train.shape, y_train.shape)
print("Validation shape:", x_val.shape, y_val.shape)
print("Test shape:", x_test.shape, y_test.shape)


DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'

## Scoring Method

In [ ]:
import math

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score


def scores(y_true, y_pred):
    return {
        "rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "R2_score": r2_score(y_true, y_pred),
        "R": np.corrcoef(np.asarray(y_true).ravel(), np.asarray(y_pred).ravel())[0, 1],
    }


## Build the Model

In [ ]:
# define neuron amounts and epoch amounts
neuron_options = [10, 20, 50, 100, 200, 300, 400, 500]
epoch_options = [10, 20, 50, 100, 200, 300, 400, 500]


def build_model(neurons, input_features, learning_rate=0.001):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_features,)),
        tf.keras.layers.Dense(neurons, activation="relu"),
        tf.keras.layers.Dense(1, activation="linear"),
    ])

    model.compile(
        loss="mean_squared_error",
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    )

    return model


## Run the model

In [ ]:
# Model training helper is defined in the next cell.


## Run Method

In [ ]:
def run_one_model(neurons, epochs, learning_rate=0.001, batch_size=32):
    tf.keras.backend.clear_session()
    model = build_model(
        neurons=neurons,
        input_features=x_train.shape[1],
        learning_rate=learning_rate,
    )

    start_time = time.time()
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
    )
    elapsed_time = time.time() - start_time

    y_train_pred = model.predict(x_train, verbose=0).ravel()
    y_val_pred = model.predict(x_val, verbose=0).ravel()
    y_test_pred = model.predict(x_test, verbose=0).ravel()

    val_scores = scores(y_val, y_val_pred)
    test_scores = scores(y_test, y_test_pred)

    return {
        "neurons": neurons,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "elapsed_time": elapsed_time,
        "history": history.history,
        "model": model,
        "train_predictions": y_train_pred,
        "val_predictions": y_val_pred,
        "test_predictions": y_test_pred,
        "val_rmse": val_scores["rmse"],
        "val_mape": val_scores["mape"],
        "val_R2_score": val_scores["R2_score"],
        "val_R": val_scores["R"],
        "test_rmse": test_scores["rmse"],
        "test_mape": test_scores["mape"],
        "test_R2_score": test_scores["R2_score"],
        "test_R": test_scores["R"],
    }


def run_all_models(neuron_options, epoch_options, learning_rate=0.001, batch_size=32):
    all_results = []
    best_result = None

    for neurons in neuron_options:
        for epochs in epoch_options:
            result = run_one_model(
                neurons=neurons,
                epochs=epochs,
                learning_rate=learning_rate,
                batch_size=batch_size,
            )
            all_results.append(result)

            if best_result is None or result["val_rmse"] < best_result["val_rmse"]:
                best_result = result

    results_df = pd.DataFrame([
        {
            "neurons": r["neurons"],
            "epochs": r["epochs"],
            "learning_rate": r["learning_rate"],
            "batch_size": r["batch_size"],
            "elapsed_time": r["elapsed_time"],
            "val_rmse": r["val_rmse"],
            "val_mape": r["val_mape"],
            "val_R2_score": r["val_R2_score"],
            "val_R": r["val_R"],
            "test_rmse": r["test_rmse"],
            "test_mape": r["test_mape"],
            "test_R2_score": r["test_R2_score"],
            "test_R": r["test_R"],
        }
        for r in all_results
    ]).sort_values(["neurons", "epochs"]).reset_index(drop=True)

    best_by_neuron_df = (
        results_df.sort_values(["neurons", "val_rmse", "epochs"])
        .groupby("neurons", as_index=False)
        .first()
        .sort_values("neurons")
        .reset_index(drop=True)
    )

    return all_results, results_df, best_by_neuron_df, best_result


## Run the models

In [ ]:
all_results, all_results_df, best_by_neuron_df, best_result = run_all_models(
    neuron_options=neuron_options,
    epoch_options=epoch_options,
    learning_rate=0.001,
    batch_size=32,
)

all_results_df.to_csv(output_dir / "all_model_results.csv", index=False)
best_by_neuron_df.to_csv(output_dir / "best_models_by_neuron.csv", index=False)
pd.DataFrame([{
    "neurons": best_result["neurons"],
    "epochs": best_result["epochs"],
    "learning_rate": best_result["learning_rate"],
    "batch_size": best_result["batch_size"],
    "elapsed_time": best_result["elapsed_time"],
    "val_rmse": best_result["val_rmse"],
    "val_mape": best_result["val_mape"],
    "val_R2_score": best_result["val_R2_score"],
    "val_R": best_result["val_R"],
    "test_rmse": best_result["test_rmse"],
    "test_mape": best_result["test_mape"],
    "test_R2_score": best_result["test_R2_score"],
    "test_R": best_result["test_R"],
}]).to_csv(output_dir / "overall_best_model_summary.csv", index=False)

pd.DataFrame({"y_train": y_train}).to_csv(output_dir / "y_train.csv", index=False)
pd.DataFrame({"y_test": y_test}).to_csv(output_dir / "y_test.csv", index=False)
pd.DataFrame({"train_prediction": best_result["train_predictions"]}).to_csv(
    output_dir / "overall_best_train_predictions.csv", index=False
)
pd.DataFrame({"test_prediction": best_result["test_predictions"]}).to_csv(
    output_dir / "overall_best_test_predictions.csv", index=False
)

for result in all_results:
    run_name = f"neurons_{result['neurons']}_epochs_{result['epochs']}"
    pd.DataFrame(result["history"]).to_csv(output_dir / f"{run_name}_history.csv", index=False)

for _, row in best_by_neuron_df.iterrows():
    result = next(
        r for r in all_results
        if r["neurons"] == row["neurons"] and r["epochs"] == row["epochs"]
    )
    pd.DataFrame(result["history"]).to_csv(
        output_dir / f"best_neuron_{int(row['neurons'])}_history.csv",
        index=False,
    )

print(best_by_neuron_df)
print("Overall best model:")
print(pd.DataFrame([{
    "neurons": best_result["neurons"],
    "epochs": best_result["epochs"],
    "val_rmse": best_result["val_rmse"],
    "test_rmse": best_result["test_rmse"],
}]))


In [ ]:
def save_best_rmse_line_plot(best_by_neuron_df, output_path):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(
        best_by_neuron_df["neurons"],
        best_by_neuron_df["val_rmse"],
        marker="o",
        linewidth=2.5,
        color="tab:blue",
    )
    ax.set_title("Best Validation RMSE by Neuron Count")
    ax.set_xlabel("Neurons")
    ax.set_ylabel("Validation RMSE")
    ax.set_xticks(best_by_neuron_df["neurons"])
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.show()


def save_metric_bar_chart(best_by_neuron_df, metric, ylabel, output_path):
    fig, ax = plt.subplots(figsize=(10, 6))
    labels = best_by_neuron_df["neurons"].astype(str)
    ax.bar(labels, best_by_neuron_df[metric], color="tab:orange")
    ax.set_title(f"Best Models by Neuron Count: {ylabel}")
    ax.set_xlabel("Neurons")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.show()


def save_loss_plots_by_neuron(all_results, output_dir):
    for neurons in sorted({result["neurons"] for result in all_results}):
        neuron_runs = sorted(
            [result for result in all_results if result["neurons"] == neurons],
            key=lambda item: item["epochs"],
        )
        fig, ax = plt.subplots(figsize=(10, 6))
        for result in neuron_runs:
            loss_history = result["history"]["loss"]
            epochs_axis = np.arange(1, len(loss_history) + 1)
            ax.plot(
                epochs_axis,
                loss_history,
                linewidth=2,
                label=f"{result['epochs']} epochs",
            )

        ax.set_title(f"Training Loss for {neurons}-Neuron Models")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(title="Run length", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(output_dir / f"loss_curves_neurons_{neurons}.png", dpi=300)
        plt.show()


save_best_rmse_line_plot(best_by_neuron_df, output_dir / "best_models_rmse_by_neuron.png")
save_metric_bar_chart(best_by_neuron_df, "val_mape", "Validation MAPE", output_dir / "best_models_mape_bar.png")
save_metric_bar_chart(best_by_neuron_df, "val_R2_score", "Validation R2 Score", output_dir / "best_models_r2_bar.png")
save_metric_bar_chart(best_by_neuron_df, "val_R", "Validation R", output_dir / "best_models_r_bar.png")
save_metric_bar_chart(best_by_neuron_df, "elapsed_time", "Elapsed Time (s)", output_dir / "best_models_elapsed_time_bar.png")
save_loss_plots_by_neuron(all_results, output_dir)
